In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import os

from tqdm import tqdm
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, separate, InMemoryDataset
from torch_geometric.utils import to_dense_batch, dense_to_sparse
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, GraphConv
from torch.utils.data import IterableDataset

from SpatialTemporalEncoder import SpatialTemporalEncoder

# Have to rename or else it conflicts with PyG imports
from types import SimpleNamespace as ns


In [35]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        ec = torch.load(os.path.join(path, subject, 'EC_geometric.pt'))
        eo = torch.load(os.path.join(path, subject, 'EO_geometric.pt'))

        ec_indices = len(ec[0].x)
        eo_indices = len(eo[0].x)


        # Break up collated data into individual samples
        # Note that separate is undocumented and I'm not sure what it precisely does
        # but it seems to work and satisfy our use case 
        for i in range(ec_indices):
            ecs.append(separate.separate(cls=ec[0].__class__, batch=ec[0], idx=i, slice_dict=ec[1], decrement=False))

        if eo[1] is None:
            eos.append(eo[0])
        else:
            for i in range(eo_indices):
                eos.append(separate.separate(cls=eo[0].__class__, batch=eo[0], idx=i, slice_dict=eo[1], decrement=False))
    dataset = ecs + eos
    flattened_dataset = []
    for data in dataset:
        data.edge_attr = data.edge_attr.flatten()[:, None]
        flattened_dataset.append(data)
    return dataset

class CustomDataset(InMemoryDataset):
    def __init__(self, listOfDataObjects):
        super().__init__()
        self.data, self.slices = self.collate(listOfDataObjects)
    
    def __len__(self):
        return len(self.slices)
    
    def __getitem__(self, idx):
        sample = self.get(idx)
        return sample

In [36]:
datasetList = load_dataset('/home/azureuser/mycontainer/SmallSubset')
print(len(datasetList))
dataset = CustomDataset(datasetList)
loader = DataLoader(datasetList, batch_size=4, shuffle=True)

  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_118098/2900805418.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ec = torch.load(os.path.join(path, subject, 'EC_geometric.pt'))
/tmp/ipykernel_118098/29

80


In [41]:
print(loader.dataset[0])
print(loader.dataset[0].edge_index.max(), loader.dataset[0].edge_index.min())
out = next(iter(loader))
print(out)
print(out.ptr)
print(torch.tensor(np.vstack(out.x)).shape, torch.tensor(np.vstack(out.x)).dtype)
print(len(out.edge_attr), out.edge_attr[0].shape, out.edge_attr[0].dtype)
print(torch.tensor(np.stack(out.edge_attr)).shape)
print(torch.flatten(torch.tensor(np.stack(out.edge_attr)), start_dim=0, end_dim=1).shape)

Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[361, 1], y=[0], pos=[19, 3])
tensor(18) tensor(0)
DataBatch(x=[4], edge_index=[2, 1444], edge_attr=[4], y=[4], pos=[76, 3], batch=[76], ptr=[5])
tensor([ 0, 19, 38, 57, 76])
torch.Size([76, 15360]) torch.float64
4 (361, 1) float64
torch.Size([4, 361, 1])
torch.Size([1444, 1])


In [6]:
configs = ns(
	seq_len = 15360,
	pred_len = 15360,
	top_k = 5,
	d_model = 19,
	d_ff = 9,
	num_kernels = 3,
	heads = 4,
	dropout = 0.1,
)

model = SpatialTemporalEncoder(configs)

In [48]:
for data in loader:
	output = model(data)
	break

torch.Size([4, 19, 15360])
